# Expand

- Original source: [documentation:ref:expand](http://redberry.cc/documentation:ref:expand)
- Back to local index: [Redberry Documentation](../Redberry.Documentation.ipynb)


In [ ]:
#r "../../NRedberry.Core/bin/Debug/net10.0/NRedberry.Core.dll"
#r "../../NRedberry.Physics/bin/Debug/net10.0/NRedberry.Physics.dll"

using NRedberry;
using NRedberry.Tensors;
using TensorFactory = NRedberry.Tensors.Tensors;


---

### Description

- `Expand` expands out products and positive integer powers.

- `Expand[transformations]` expands out products and positive integer powers and applies `transformations` at each level of expand procedure.

### Examples

---

Expand a polynomial expressions:
```groovy
println Expand >> '(x_n + y_n)*(f_m - r_m)'.t
```

```text
   > x_{n}*f_{m}+f_{m}*y_{n}-r_{m}*y_{n}-x_{n}*r_{m}
```


```groovy
println Expand >> '(1 + x)**4'.t
```

```text
   > x**4+1+4*x**3+6*x**2+4*x
```


---

```groovy
println Expand >> '(x + y)/z'.t
```

```text
   > x/z+y/z
```


---

`Expand` relabels dummies when necessary:
```groovy
println Expand >> '(A_m^m + 1)**3'.t
```

```text
   > 3*A_{m}^{m}*A_{a}^{a}+A_{m}^{m}*A_{a}^{a}*A_{b}^{b}+1+3*A_{b}^{b}
```



---



`Expand` does not go inside functions and denominators; `ExpandAll` does:
```groovy
println Expand >> 'f[(x + y)**2]'.t
```

```text
   > f[(x + y)**2]
```

```groovy
println ExpandAll >> 'f[(x + y)**2]'.t
```

```text
   > f[x**2 + 2*x*y + y**2]
```


---

### Details

`Expand[transformations]` will additionally apply `transformations` during expand procedure:
```groovy
println Expand['k_a*k^a = 0'.t] >> '(k_a + t_a)*(k^a + t^a)'.t
```

```text
   > 2*k_a*t^a + t_a*t^a
```


Passing additional `transformations` can significantly improve the performance when expanding huge expressions. For example, when a huge expression involves many metric tensors, one can pass `EliminateMetrics` in order to reduce the number of processed terms. Consider a random example:
```groovy
//create random generator, which generates
// random tensors consisting of metric and A_m
RandomTensor randomTensor = new RandomTensor();
randomTensor.clearNamespace()
randomTensor.addToNamespace('g_mn'.t, 'A_m'.t)

//loop to warm up JVM
for (def i in 1..1000) {
    def a, b
    //next random tensor
    def t = randomTensor.nextTensorTree(4, 3, 8, '_a'.si)
    def simplify = EliminateMetrics & 'A_a*A^a = 1'.t & 'd^i_i = 10'.t

    //this will typically 10 times faster
    timing {
        a = Expand[simplify] >> t
    }
    //then this
    timing {
        b = (Expand & simplify) >> t
    }

    assert a == b
    println ''
}
```

The sample output will looks like:
```text
Time: 10 ms.
Time: 1015 ms.

Time: 7 ms.
Time: 6566 ms.

Time: 66 ms.
Time: 983 ms.
...
```


### See also

- Related guides: [applying and manipulating transformations](../guide/ApplyingAndManipulatingTransformations.ipynb), [list of transformations](../guide/ListOfTransformations.ipynb)
- Related transformations: ExpandTensors, ExpandAll, ExpandNumerator, ExpandDenominator
- JavaDocs: [ExpandTransformation](http://api.redberry.cc/redberry/1.1.9/java-api/cc/redberry/core/transformations/expand/ExpandTransformation.html)
- Source code: [ExpandTransformation.java](https://bitbucket.org/redberry/redberry/src/tip/core/src/main/java/cc/redberry/core/transformations/expand/ExpandTransformation.java)
